In [1]:
import os
import sys
import subprocess
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torchvision.models import ResNet50_Weights
from torch.utils.data import DataLoader, Subset
import numpy as np
from tqdm import tqdm
import tarfile
from transformers import AutoImageProcessor, AutoModel
import random


In [2]:
# noinspection JupyterPackage
import urllib

# --- 1. ROZWIĄZANIE KONFLIKTÓW SYSTEMOWYCH ---
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 45390
# --- 2. KONFIGURACJA REPOZYTORIUM ---
REPO_NAME = "ssl-data-curation"
REPO_URL = "https://github.com/facebookresearch/ssl-data-curation.git"


#
def setup_repo():
    if not os.path.exists(REPO_NAME):
        print(f"--- Klonowanie repozytorium {REPO_NAME}... ---")
        subprocess.run(["git", "clone", REPO_URL], check=True)

    # Dodanie katalogów do sys.path, aby Python widział folder 'src'
    repo_path = os.path.abspath(REPO_NAME)
    src_path = os.path.join(repo_path, "src")

    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)
    print("--- Ścieżki repozytorium skonfigurowane ---")


#
setup_repo()
# # --- 3. IMPORTY Z REPOZYTORIUM (PO USTAWIENIU ŚCIEŻEK) ---
try:
    from src.clusters import HierarchicalCluster
    from src import hierarchical_kmeans_gpu as hkmg
    from src import hierarchical_sampling as hs

    print("--- Moduły Facebook Research załadowane pomyślnie ---")
except ImportError as e:
    print(f"Błąd importu: {e}")
    sys.exit(1)
#
# # --- 4. PRZYGOTOWANIE DANYCH (IMAGENETTE) ---
DATA_URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz"
DATA_DIR = "imagenette2-320"


#
#
def prepare_data():
    if not os.path.exists(DATA_DIR):
        print("--- Pobieranie zbioru Imagenette (320px)... ---")
        if not os.path.exists("imagenette.tgz"):
          urllib.request.urlretrieve(DATA_URL, "imagenette.tgz")
        with tarfile.open("imagenette.tgz", "r:gz") as tar:
            tar.extractall(path=DATA_DIR)


# --- 5. GŁÓWNA FUNKCJA TRENINGOWA ---
def train_model(model, train_loader, test_loader, device, title, epochs=5):
    print(f"\n[TRENING] Start: {title}")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    skipped_steps_counter = 0
    scaler = torch.amp.GradScaler("cuda")
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for imgs, labels in tqdm(train_loader, desc=f"Epoka {epoch + 1}/{epochs}"):
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none = True)
            with torch.amp.autocast("cuda"):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()

            #makeing a step and checking wheter scaler made fp16 overflow
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            new_scale = scaler.get_scale()
            if old_scale != new_scale:
                skipped_steps_counter+=1
                print(f"Step skipped, scale reduced from {old_scale} -> {new_scale} (batch was futile)")
            running_loss += loss.item()

    # Ewaluacja
    model.eval()
    correct = 0
    total = 0
    with torch.inference_mode():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"[WYNIK] {title} Accuracy: {accuracy:.2f}%")
    return accuracy


def get_resnet50_embedings(dataset):
    extractor = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2).to(device)
    extractor.fc = nn.Identity()
    extractor.eval()
    
    all_features = []
    with torch.inference_mode(), torch.amp.autocast("cuda"):
        feature_loader = DataLoader(dataset, batch_size=128, shuffle=False)
        for imgs, _ in tqdm(feature_loader, desc="Ekstrakcja cech"):
            feat = extractor(imgs.to(device))
            all_features.append(feat.cpu())            
    
    return torch.cat(all_features).to(device)

class DinoTransform:
    def __init__(self, processor= AutoImageProcessor.from_pretrained('facebook/dinov2-large') ):
        self.processor = processor
    def __call__(self, img):
        return self.processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0)

def get_dino_embedings(dataset):
    #dataset need proper tronsform (DinoTransform)
    extr_loader = DataLoader(dataset,
                             batch_size=64,
                             shuffle=False,
                             num_workers=0) #można 2 w colabie albo na linuxie

    all_embedings = []
    extractor = AutoModel.from_pretrained('facebook/dinov2-large')
    extractor.eval()
    extractor.to(device)
    with torch.inference_mode(), torch.amp.autocast("cuda"):
        for imgs,_ in tqdm(extr_loader, desc="Wyciągnie embedingów: "):
            imgs = imgs.to(device)
            outputs = extractor(imgs)
            embedings = outputs.last_hidden_state[:,0,:]
            all_embedings.append(embedings.cpu())

    return torch.cat(all_embedings).to(device)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

--- Ścieżki repozytorium skonfigurowane ---
--- Moduły Facebook Research załadowane pomyślnie ---


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

D:\programy_studia\wb2\WB2_Project\.venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\micae\.cache\huggingface\hub\models--facebook--dinov2-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [5]:
# --- 6. URUCHOMIENIE EKSPERYMENTU ---
prepare_data()
print(f"Praca na urządzeniu: {device}")

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
real_data_dir = os.path.join("imagenette2-320")
full_train_dataset = datasets.ImageFolder(os.path.join(real_data_dir, 'train'), transform=transform)
test_dataset = datasets.ImageFolder(os.path.join(real_data_dir, 'val'), transform=transform)

test_loader = DataLoader(test_dataset,
                            batch_size=64,
                            shuffle=False,
                            num_workers=2)

print(f"--- TRYB PEŁNY: Zbiór treningowy liczy {len(full_train_dataset)} obrazów ---")

# EKSTRAKCJA CECH (Dla wszystkich obrazów w zbiorze)
print("\n--- KROK 1: Ekstrakcja cech dla całego zbioru ---")

# code to get embedings from dino for sampling purposes
# dataset_dino = datasets.ImageFolder(os.path.join(real_data_dir,"train"), transform=DinoTransform())
# data_tensor = get_dino_embedings(dataset_dino)
set_seed(SEED)
data_tensor = get_resnet50_embedings(full_train_dataset).float()

print("\n--- KROK 2: Hierarchiczny kmeans---")
set_seed(SEED)
clusters = hkmg.hierarchical_kmeans_with_resampling(
    data=data_tensor,                                                                           
    n_clusters=[120, 50],
    n_levels=2,
    sample_sizes=[15, 2],
    verbose=False,
)

cl = HierarchicalCluster.from_dict(clusters)

# target_subset_size = 8000
# sampled_indices = hs.hierarchical_sampling(cl, target_size=target_subset_size)

# sampled_dataset = Subset(full_train_dataset, sampled_indices)
# print(f"Wyselekcjonowano {len(sampled_indices)} obrazów za pomocą Twojej metody.")

# num_epochs = 10

# model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
# model.fc = nn.Linear(model.fc.in_features, 10)
# model = model.to(device)
# acc_sampled = train_model(
#     model,
#     DataLoader(sampled_dataset, batch_size=64, shuffle=True, num_workers=2, persistent_workers=True),
#     test_loader,
#     device,
#     f"RANKING SUBSET ({target_subset_size} images)",
#     epochs=num_epochs
# )

# model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
# model.fc = nn.Linear(model.fc.in_features, 10)
# model = model.to(device)
# acc_full = train_model(
#     model,
#     DataLoader(full_train_dataset, batch_size=64, shuffle=True, num_workers=2),
#     test_loader,
#     device,
#     f"FULL DATASET ({len(full_train_dataset)} images)",
#     epochs=num_epochs
# )

# print("\n" + "=" * 50)
# print(f"FINALNE PODSUMOWANIE PO {num_epochs} EPOKACH:")
# print(f"Accuracy - Pełny zbiór: {acc_full:.2f}%")
# print(f"Accuracy - Twoja metoda (tylko {target_subset_size} zdjęć): {acc_sampled:.2f}%")

# efektywnosc = acc_sampled / acc_full * 100
# print(f"Twoja metoda osiągnęła {efektywnosc:.1f}% jakości pełnego zbioru, "
#         f"używając jedynie {target_subset_size / len(full_train_dataset) * 100:.1f}% danych!")
# print("=" * 50)


Praca na urządzeniu: cuda
--- TRYB PEŁNY: Zbiór treningowy liczy 9469 obrazów ---

--- KROK 1: Ekstrakcja cech dla całego zbioru ---


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\micae/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:09<00:00, 10.7MB/s]
Ekstrakcja cech: 100%|██████████| 74/74 [00:37<00:00,  1.99it/s]



--- KROK 2: Hierarchiczny kmeans---
Hierarchical k-means resampling steps: 100%|██████████| 10/10 [00:05<00:00,  1.83it/s]


In [4]:
num_epochs = 10
set_seed(SEED)
model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)
acc_full = train_model(
    model,
    DataLoader(full_train_dataset, batch_size=64, shuffle=True, num_workers=2, generator=torch.Generator().manual_seed(SEED)),
    test_loader,
    device,
    f"FULL DATASET ({len(full_train_dataset)} images)",
    epochs=num_epochs,
)

subset_sizes = [500*i for i in range(1,int(np.ceil(len(full_train_dataset)/500)))]
acc_sampled_list = []
for target_subset_size in subset_sizes:
    set_seed(SEED)#chociaż nie wiem czy potrzeba to chyba deterministyczne
    sampled_indices = hs.hierarchical_sampling(cl, target_size=target_subset_size)
    sampled_dataset = Subset(full_train_dataset, sampled_indices)
    print(f"Wyselekcjonowano {len(sampled_indices)} obrazów za pomocą Twojej metody.")

    set_seed(SEED)
    model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Linear(model.fc.in_features, 10)
    model = model.to(device)
    acc_sampled = train_model(
        model,
        DataLoader(sampled_dataset, batch_size=64, shuffle=True, num_workers=2, persistent_workers=True, generator=torch.Generator().manual_seed(SEED)),
        test_loader,
        device,
        f"RANKING SUBSET ({target_subset_size} images)",
        epochs=num_epochs
    )
    acc_sampled_list.append(acc_sampled)



print("\n" + "=" * 50)
print(f"FINALNE PODSUMOWANIE PO SPRAWDZENIU {len(subset_sizes)} SUBSETÓW:")
print("-"*50)
print(f"Pełny zbiór: Accuracy - {acc_full:.2f}%")
print("-"*50)
for i in range(len(subset_sizes)):
    print(f"{subset_sizes[i]} zdjęć: Accuracy - {acc_sampled_list[i]:.2f}% | Efektywność - {(acc_sampled_list[i] / acc_full) * 100:.2f}% {'!!!!' if (acc_sampled_list[i] / acc_full)>1 else ''}")
print("-"*50)
print("Najlepszy wynik uzyskał:")
if np.max(acc_sampled_list) >= acc_full:
    i = np.argmax(acc_sampled_list)
    print(f"{subset_sizes[i]} zdjęć: Accuracy - {acc_sampled_list[i]:.2f}% | Efektywność - {(acc_sampled_list[i] / acc_full) * 100:.2f}% {'!!!!' if (acc_sampled_list[i] / acc_full)>1 else ''}")
else:
    print(f"Pełny zbiór: Accuracy - {acc_full:.2f}%")




[TRENING] Start: FULL DATASET (9469 images)


Epoka 10/10: 100%|██████████| 148/148 [00:33<00:00,  4.39it/s]


[WYNIK] FULL DATASET (9469 images) Accuracy: 92.03%
Hierarchical sampling from clusters: 100%|██████████| 50/50 [00:00<00:00, 13850.82it/s]
Wyselekcjonowano 500 obrazów za pomocą Twojej metody.

[TRENING] Start: RANKING SUBSET (500 images)


Epoka 10/10: 100%|██████████| 8/8 [00:01<00:00,  5.02it/s]


[WYNIK] RANKING SUBSET (500 images) Accuracy: 56.10%
Hierarchical sampling from clusters: 100%|██████████| 50/50 [00:00<00:00, 15268.67it/s]
Wyselekcjonowano 1000 obrazów za pomocą Twojej metody.

[TRENING] Start: RANKING SUBSET (1000 images)


...

[WYNIK] RANKING SUBSET (8500 images) Accuracy: 89.12%
Hierarchical sampling from clusters: 100%|██████████| 50/50 [00:00<00:00, 22857.24it/s]
Wyselekcjonowano 9000 obrazów za pomocą Twojej metody.

[TRENING] Start: RANKING SUBSET (9000 images)


Epoka 1/10:  35%|███▌      | 50/141 [00:14<00:15,  6.06it/s]

Step skipped, scale reduced from 65536.0 -> 32768.0 (batch was futile)


Epoka 10/10: 100%|██████████| 141/141 [00:23<00:00,  5.97it/s]


[WYNIK] RANKING SUBSET (9000 images) Accuracy: 90.55%

FINALNE PODSUMOWANIE PO SPRAWDZENIU 18 SUBSETÓW:
--------------------------------------------------
Pełny zbiór: Accuracy - 92.03%
--------------------------------------------------
500 zdjęć: Accuracy - 56.10% | Efektywność - 60.96% 
1000 zdjęć: Accuracy - 76.76% | Efektywność - 83.42% 
1500 zdjęć: Accuracy - 79.13% | Efektywność - 85.99% 
2000 zdjęć: Accuracy - 77.40% | Efektywność - 84.11% 
2500 zdjęć: Accuracy - 82.70% | Efektywność - 89.87% 
3000 zdjęć: Accuracy - 82.27% | Efektywność - 89.40% 
3500 zdjęć: Accuracy - 86.29% | Efektywność - 93.77% 
4000 zdjęć: Accuracy - 83.49% | Efektywność - 90.73% 
4500 zdjęć: Accuracy - 80.33% | Efektywność - 87.29% 
5000 zdjęć: Accuracy - 83.54% | Efektywność - 90.78% 
5500 zdjęć: Accuracy - 87.49% | Efektywność - 95.07% 
6000 zdjęć: Accuracy - 88.18% | Efektywność - 95.82% 
6500 zdjęć: Accuracy - 88.69% | Efektywność - 96.37% 
7000 zdjęć: Accuracy - 90.62% | Efektywność - 98.48% 
7500 zdj

In [7]:
# --- KROK 1: Przygotowanie transformacji i modelu DINOv2 ---
print("\n--- KROK 1: Ekstrakcja embedingów za pomocą DINOv2 ---")

# Ustawiamy ziarno dla powtarzalności
set_seed(SEED)

# Specjalna transformacja dla DINOv2 (używa procesora od Hugging Face)
dino_transform = DinoTransform()

# Tworzymy tymczasowy dataset z transformacją DINO
dataset_dino = datasets.ImageFolder(os.path.join(real_data_dir, "train"), transform=dino_transform)

# Wyciągamy embedingi (to może chwilę potrwać w zależności od GPU)
# DINOv2-large ma 1024 wymiary, co świetnie nadaje się do klastrowania
data_tensor_dino = get_dino_embedings(dataset_dino).float()

print(f"Wyciągnięto embedingi o kształcie: {data_tensor_dino.shape}")

# --- KROK 2: Hierarchiczny K-Means na bazie DINO ---
print("\n--- KROK 2: Hierarchiczny kmeans (Facebook Research) ---")
set_seed(SEED)

clusters_dino = hkmg.hierarchical_kmeans_with_resampling(
    data=data_tensor_dino,                                                                           
    n_clusters=[120, 50],
    n_levels=2,
    sample_sizes=[15, 2],
    verbose=False,
)

cl_dino = HierarchicalCluster.from_dict(clusters_dino)

# --- KROK 3: Samplowanie i Trening ---
target_subset_size = 8000
sampled_indices = hs.hierarchical_sampling(cl_dino, target_size=target_subset_size)

# Tworzymy subset korzystając z oryginalnego full_train_dataset (z normalnymi transformacjami!)
sampled_dataset = Subset(full_train_dataset, sampled_indices)
print(f"Wyselekcjonowano {len(sampled_indices)} obrazów za pomocą DINOv2 + Hierarchical Sampling.")

num_epochs = 10

# 1. Trening na podzbiorze (Sampled)
model_sampled = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
model_sampled.fc = nn.Linear(model_sampled.fc.in_features, 10)
model_sampled = model_sampled.to(device)

acc_sampled = train_model(
    model_sampled,
    DataLoader(sampled_dataset, batch_size=64, shuffle=True, num_workers=2, persistent_workers=True),
    test_loader,
    device,
    f"DINO SAMPLED SUBSET ({target_subset_size} images)",
    epochs=num_epochs
)

# 2. Trening na pełnym zbiorze (Dla porównania)
model_full = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
model_full.fc = nn.Linear(model_full.fc.in_features, 10)
model_full = model_full.to(device)

acc_full = train_model(
    model_full,
    DataLoader(full_train_dataset, batch_size=64, shuffle=True, num_workers=2),
    test_loader,
    device,
    f"FULL DATASET ({len(full_train_dataset)} images)",
    epochs=num_epochs
)

# --- KROK 4: Finalny Raport ---
print("\n" + "=" * 50)
print(f"PODSUMOWANIE EKSPERYMENTU (DINOv2):")
print(f"Accuracy - Pełny zbiór ({len(full_train_dataset)} zdjęć): {acc_full:.2f}%")
print(f"Accuracy - Metoda z DINOv2 ({target_subset_size} zdjęć): {acc_sampled:.2f}%")

efektywnosc = acc_sampled / acc_full * 100
procent_danych = (target_subset_size / len(full_train_dataset)) * 100
print(f"Efektywność: {efektywnosc:.1f}% jakości przy użyciu {procent_danych:.1f}% danych.")
print("=" * 50)


--- KROK 1: Ekstrakcja embedingów za pomocą DINOv2 ---


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

Wyciągnie embedingów:   0%|          | 0/148 [00:00<?, ?it/s]D:\programy_studia\wb2\WB2_Project\.venv\lib\site-packages\transformers\integrations\sdpa_attention.py:92: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Wyciągnie embedingów: 100%|██████████| 148/148 [02:16<00:00,  1.09it/s]


Wyciągnięto embedingi o kształcie: torch.Size([9469, 1024])

--- KROK 2: Hierarchiczny kmeans (Facebook Research) ---
Hierarchical sampling from clusters: 100%|██████████| 50/50 [00:00<00:00, 9997.86it/s]
Wyselekcjonowano 8000 obrazów za pomocą DINOv2 + Hierarchical Sampling.

[TRENING] Start: DINO SAMPLED SUBSET (8000 images)


Epoka 10/10: 100%|██████████| 125/125 [00:31<00:00,  3.94it/s]


[WYNIK] DINO SAMPLED SUBSET (8000 images) Accuracy: 89.99%

[TRENING] Start: FULL DATASET (9469 images)


Epoka 10/10: 100%|██████████| 148/148 [00:40<00:00,  3.64it/s]


[WYNIK] FULL DATASET (9469 images) Accuracy: 90.47%

PODSUMOWANIE EKSPERYMENTU (DINOv2):
Accuracy - Pełny zbiór (9469 zdjęć): 90.47%
Accuracy - Metoda z DINOv2 (8000 zdjęć): 89.99%
Efektywność: 99.5% jakości przy użyciu 84.5% danych.
